# Multi-Label Text Classification (BertForSequenceClassification) (used)

In [55]:
import pandas as pd
import ast
from sklearn.model_selection import train_test_split
from transformers import BertTokenizerFast
import torch
from torch.utils.data import Dataset
from transformers import BertForSequenceClassification
from torch.utils.data import DataLoader
from transformers import AdamW
from tqdm import tqdm
import numpy as np





In [56]:
df = pd.read_csv("ARP_PreLabel.csv")

def extract_entity_names(entities_str):
    try:
        entities = ast.literal_eval(entities_str)
        return [e[0] for e in entities if isinstance(e, tuple)]
    except:
        return []

df["Entity_Texts"] = df["Entities"].apply(extract_entity_names)


In [57]:
ENTITY_LIST = [
    "Federal Reserve", "Interest Rates", "Inflation", "Employment", "Unemployment", "GDP", "Trade", "Congress", "Monetary Policy", "Financial Stability", 
    "Price Stability", "Regulatory Implementation", "Pandemic", "Asset Runoff", "Reinvestment", "Money Market", "Bond Market", "Equity Markets", "Financial Markets", "Repo Markets", 
    "Fiscal Policy", "Balance Sheet", "Reserves", "Digital Dollar", "Foreign Currencies", "Federal Funds", "Demand", "Securities", "War", "Finance", 
    "Debt", "Mortgage", "Maturity", "Credit", "Labor Market", "Auction", "Press Conference", "Banking System", "Uncertain", "Development", "Economic Outlook", "Countries"
]


In [58]:
# label vector (0 or 1) for each sentence
def label_vector_from_entities(entity_names):
    vec = [0] * len(ENTITY_LIST)
    for i, ent in enumerate(ENTITY_LIST):
        if ent in entity_names:
            vec[i] = 1
    return vec

df["Label_Vector"] = df["Entity_Texts"].apply(label_vector_from_entities)


In [59]:
train_df, test_df = train_test_split(df, test_size=0.3, random_state=123)

In [60]:
# converted to strings
train_df["Sentence"] = train_df["Sentence"].astype(str)
test_df["Sentence"]  = test_df["Sentence"].astype(str)

In [61]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-cased")

# encode
train_encodings = tokenizer(train_df["Sentence"].tolist(), truncation=True, padding=True, max_length=128)
test_encodings  = tokenizer(test_df["Sentence"].tolist(),  truncation=True, padding=True, max_length=128)

train_labels = train_df["Label_Vector"].tolist()
test_labels  = test_df["Label_Vector"].tolist()


In [62]:

class MultiLabelDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
        
    def __getitem__(self, idx):
        return {
            key: torch.tensor(val[idx]) for key, val in self.encodings.items()
        } | {
            "labels": torch.tensor(self.labels[idx], dtype=torch.float)
        }

    def __len__(self):
        return len(self.labels)

train_dataset = MultiLabelDataset(train_encodings, train_labels)
test_dataset  = MultiLabelDataset(test_encodings, test_labels)


In [63]:

model = BertForSequenceClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(ENTITY_LIST),
    problem_type="multi_label_classification"
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [38]:
# prepare training data & device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
optimizer = AdamW(model.parameters(), lr=2e-5)

# calculate pos_weight
label_matrix = np.array(train_labels)  # shape: [num_samples, 42]
pos_counts = label_matrix.sum(axis=0)
neg_counts = len(label_matrix) - pos_counts
pos_weight = torch.tensor(neg_counts / (pos_counts + 1e-5), dtype=torch.float).to(device)

# initialise the weighted loss function
loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

model.train()
for epoch in range(30):
    total_loss = 0
    print(f"Epoch {epoch+1}")
    
    for batch in tqdm(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()
    
    print(f"Avg loss: {total_loss / len(train_loader):.4f}")


/opt/anaconda3/envs/ARP/lib/python3.12/site-packages/transformers/optimization.py:588: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1


100%|██████████| 65/65 [02:04<00:00,  1.91s/it]


Avg loss: 1.2958
Epoch 2


100%|██████████| 65/65 [01:59<00:00,  1.84s/it]


Avg loss: 1.1518
Epoch 3


100%|██████████| 65/65 [01:56<00:00,  1.80s/it]


Avg loss: 1.0321
Epoch 4


  9%|▉         | 6/65 [00:12<02:00,  2.03s/it]


KeyboardInterrupt: 

In [ ]:
#model.save_pretrained(f"MicroF1_0.85")
#tokenizer.save_pretrained(f"MicroF1_0.85")

In [ ]:
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
import numpy as np

model.eval()
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=True)
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].cpu().numpy()  

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.sigmoid(outputs.logits).cpu().numpy()
        preds = (probs > 0.5).astype(int)

        all_preds.append(preds)
        all_labels.append(labels)


y_pred = np.vstack(all_preds)
y_true = np.vstack(all_labels)

In [ ]:
print("F1-Score Evaluation:")

print("Micro F1:", f1_score(y_true, y_pred, average="micro"))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))
print("Weighted F1:", f1_score(y_true, y_pred, average="weighted"))

report = classification_report(
    y_true, y_pred, target_names=ENTITY_LIST, zero_division=0
)
print(report)


In [ ]:
import torch
from transformers import BertTokenizerFast, BertForSequenceClassification
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
import numpy as np
from tqdm import tqdm


model_path = "MicroF1_0.85" 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=True)

tokenizer = BertTokenizerFast.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path)
model.to(device)
model.eval()


all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].cpu().numpy()  

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.sigmoid(outputs.logits).cpu().numpy()
        preds = (probs > 0.5).astype(int) 

        all_preds.append(preds)
        all_labels.append(labels)


y_pred = np.vstack(all_preds)
y_true = np.vstack(all_labels)

print("Micro F1:", f1_score(y_true, y_pred, average="micro"))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))
print("Precision (micro):", precision_score(y_true, y_pred, average="micro"))
print("Recall (micro):", recall_score(y_true, y_pred, average="micro"))

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=ENTITY_LIST))

# CrossValidation (test)

In [ ]:
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.metrics import f1_score
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from transformers import AdamW
from tqdm import tqdm

# 转换为 numpy 格式
all_inputs = train_encodings
all_labels = np.array(train_labels)

# 设置 5 折
mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold = 0
all_fold_f1s = []

for train_idx, val_idx in mskf.split(np.zeros(len(all_labels)), all_labels):
    fold += 1
    print(f"\n🌀 Fold {fold}")

    # 切分数据集
    train_dataset_fold = torch.utils.data.Subset(train_dataset, train_idx)
    val_dataset_fold   = torch.utils.data.Subset(train_dataset, val_idx)

    train_loader = DataLoader(train_dataset_fold, batch_size=16, shuffle=True)
    val_loader   = DataLoader(val_dataset_fold, batch_size=32)

    # 初始化模型和优化器
    model = CustomBERT(num_labels=42).to(device)
    optimizer = AdamW(model.parameters(), lr=2e-5)

    # 计算当前 fold 的 pos_weight
    y_fold = all_labels[train_idx]
    pos_counts = y_fold.sum(axis=0)
    neg_counts = len(y_fold) - pos_counts
    pos_weight = torch.tensor(neg_counts / (pos_counts + 1e-5), dtype=torch.float).to(device)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    # 训练几轮即可（如 5）
    model.train()
    for epoch in range(10):
        total_loss = 0
        for batch in tqdm(train_loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs, labels)

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total_loss += loss.item()
        print(f"Fold {fold} | Epoch {epoch+1} | Loss: {total_loss / len(train_loader):.4f}")

    # 验证集评估
    model.eval()
    all_preds, all_trues = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].cpu().numpy()

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.sigmoid(outputs).cpu().numpy()
            preds = (probs > 0.5).astype(int)

            all_preds.append(preds)
            all_trues.append(labels)

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_trues)

    f1 = f1_score(y_true, y_pred, average="micro")
    print(f"📊 Fold {fold} Micro F1: {f1:.4f}")
    all_fold_f1s.append(f1)

# 输出总体评估
print(f"Average Micro F1 across folds: {np.mean(all_fold_f1s):.4f}")



In [54]:
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
import numpy as np

model.eval()
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=True)
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].cpu().numpy()

        logits = model(input_ids=input_ids, attention_mask=attention_mask)  # ← 不要 .logits
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs > 0.5).astype(int)

        all_preds.append(preds)
        all_labels.append(labels)

y_pred = np.vstack(all_preds)
y_true = np.vstack(all_labels)


  0%|          | 0/28 [00:00<?, ?it/s]


TypeError: sigmoid(): argument 'input' (position 1) must be Tensor, not SequenceClassifierOutput

In [ ]:
print("F1-Score Evaluation:")

print("Micro F1:", f1_score(y_true, y_pred, average="micro"))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))
print("Weighted F1:", f1_score(y_true, y_pred, average="weighted"))

report = classification_report(
    y_true, y_pred, target_names=ENTITY_LIST, zero_division=0
)
print(report)


# NER (test)

In [14]:
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from tqdm import tqdm
import pandas as pd, ast
from transformers import BertTokenizerFast

In [15]:
import pandas as pd
import ast

df = pd.read_csv("ARP_PreLabel.csv")

def parse_entities(entity_str):
    try:
        parsed = ast.literal_eval(entity_str)
        return [e[0] for e in parsed if isinstance(e, tuple)]
    except:
        return []

df["Entity_Texts"] = df["Entities"].apply(parse_entities)


In [16]:
def get_char_spans(text, entity_list):
    spans = []
    for ent in entity_list:
        start = text.lower().find(ent.lower())
        if start != -1:
            end = start + len(ent)
            spans.append((start, end, "ENTITY"))
    return spans

df["Entity_Spans"] = df.apply(lambda row: get_char_spans(row["Sentence"], row["Entity_Texts"]), axis=1)


In [17]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-cased")

In [18]:
def tokenize_and_create_labels(text, spans):
    encoding = tokenizer(text, return_offsets_mapping=True, truncation=True)
    labels = ["O"] * len(encoding["offset_mapping"])
    for i, (start, end) in enumerate(encoding["offset_mapping"]):
        for span_start, span_end, _ in spans:
            if start >= span_start and end <= span_end:
                labels[i] = "B-ENTITY" if labels[i] == "O" else "I-ENTITY"
    return tokenizer.convert_ids_to_tokens(encoding["input_ids"]), labels


In [19]:
df["TokensAndLabels"] = df.apply(lambda row: tokenize_and_create_labels(row["Sentence"], row["Entity_Spans"]), axis=1)


In [20]:
df = df.dropna(subset=["Sentence", "Entity_Spans"])
df["TokensAndLabels"] = df.apply(
    lambda row: tokenize_and_create_labels(row["Sentence"], row["Entity_Spans"]),
    axis=1
)


In [21]:
from sklearn.model_selection import train_test_split

data = []
for _, row in df.iterrows():
    tokens, labels = row["TokensAndLabels"]
    data.append({"tokens": tokens, "ner_tags": labels})

train_data, test_data = train_test_split(data, test_size=0.3, random_state=123)


In [22]:
from datasets import Dataset, DatasetDict

label_list = ["O", "B-ENTITY", "I-ENTITY"]
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)
dataset = DatasetDict({"train": train_dataset, "test": test_dataset})

def align_labels(example):
    tokenized = tokenizer(example["tokens"], is_split_into_words=True, truncation=True, padding="max_length", max_length=128)
    word_ids = tokenized.word_ids()
    labels = []
    prev_word_idx = None
    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        elif word_idx != prev_word_idx:
            labels.append(label2id[example["ner_tags"][word_idx]])
        else:
            labels.append(label2id[example["ner_tags"][word_idx]])
        prev_word_idx = word_idx
    tokenized["labels"] = labels
    return tokenized

dataset = dataset.map(align_labels)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Map: 100%|██████████| 445/445 [00:00<00:00, 3469.21 examples/s]


In [23]:
from transformers import BertForTokenClassification, AdamW
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
from tqdm import tqdm

# tensor
dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# create DataLoader
train_loader = DataLoader(dataset["train"], batch_size=16, shuffle=True)
val_loader = DataLoader(dataset["test"], batch_size=32)


model = BertForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = AdamW(model.parameters(), lr=5e-5)
loss_fn = nn.CrossEntropyLoss()


model.train()
for epoch in range(3):
    total_loss = 0
    print(f"\nEpoch {epoch+1}")
    for batch in tqdm(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        # logits shape: [batch_size, seq_len, num_labels]
        # labels shape: [batch_size, seq_len]
        loss = loss_fn(logits.view(-1, model.num_labels), labels.view(-1))
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} - Loss: {avg_loss:.4f}")


Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/opt/anaconda3/envs/ARP/lib/python3.12/site-packages/transformers/optimization.py:588: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(



Epoch 1


100%|██████████| 65/65 [01:55<00:00,  1.78s/it]


Epoch 1 - Loss: 0.1680

Epoch 2


100%|██████████| 65/65 [01:55<00:00,  1.77s/it]


Epoch 2 - Loss: 0.0595

Epoch 3


100%|██████████| 65/65 [01:50<00:00,  1.70s/it]

Epoch 3 - Loss: 0.0413



save_dir = "./bert_ner_model"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)



import os
import torch
from transformers import BertTokenizerFast, BertForTokenClassification

def extract_ner_entities(sentences, model_dir="./bert_ner_model"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    
    tokenizer = BertTokenizerFast.from_pretrained(model_dir)
    model = BertForTokenClassification.from_pretrained(model_dir)
    model.to(device)
    model.eval()

    
    id2label = model.config.id2label

    results = []
    with torch.no_grad():
        encodings = tokenizer(sentences, truncation=True, padding=True, max_length=128, return_tensors="pt").to(device)
        logits = model(**encodings).logits
        preds = torch.argmax(logits, dim=-1).cpu().numpy()  # [B, T]
        tokens_batch = [tokenizer.convert_ids_to_tokens(ids) for ids in encodings["input_ids"].cpu().numpy()]

        for sent, tokens, pred_ids in zip(sentences, tokens_batch, preds):
            entities = []
            current = []
            for tok, pred_id in zip(tokens, pred_ids):
                label = id2label.get(int(pred_id), "O")
                if label.startswith("B-"):
                    if current:
                        entities.append(" ".join(current))
                    current = [tok]
                elif label.startswith("I-") and current:
                    current.append(tok)
                else:
                    if current:
                        entities.append(" ".join(current))
                        current = []
            if current:
                entities.append(" ".join(current))

            results.append({"sentence": sent, "entities": entities})

    return results
